In [1]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
with open("processed_data_pkl/imputed_training_data.pkl", "rb") as f:
    traffic_dic = pickle.load(f)

with open("processed_data_pkl/weather_global_2014_2025.pkl", "rb") as f:
    weather = pickle.load(f)

air_qual = pd.read_csv("online_data/air_qual/air_qual.csv")

In [3]:
air_qual.drop(columns=["aerosol_optical_depth ()", "dust (μg/m³)"], inplace=True)
air_qual.dropna(inplace=True)
air_qual.reset_index(inplace=True, drop=True)

In [4]:
air_qual["time"] = pd.to_datetime(air_qual["time"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather = weather[weather["timestamp"].isin(air_qual["time"])]
weather.reset_index(inplace=True, drop=True)

In [5]:
dataset = pd.concat([weather, air_qual], axis=1)
dataset.drop(columns=["time"], inplace=True)

# setting up the graph

In [6]:
import networkx as nx
import torch
from torch_geometric.utils import to_networkx
from torch_geometric.nn import GCNConv
from torch.nn import Linear
import geopy.distance
from torch_geometric.data import Data


In [7]:
G = nx.DiGraph()

for sensor_id, directions in traffic_dic.items():
    for direction, df in directions.items():
        origin_lat = df["latitude"].iloc[0]
        origin_lon = df["longitude"].iloc[0]

        G.add_node(sensor_id, pos=(origin_lon, origin_lat))
        
        for sensor_id_2, directions_2 in traffic_dic.items():
            if sensor_id == sensor_id_2:
                continue

            for direction_2, df_2 in directions_2.items():
                target_lat = df_2["latitude"].iloc[0]
                target_lon = df_2["longitude"].iloc[0]
                
                distance_m = geopy.distance.geodesic(
                    (origin_lat, origin_lon), 
                    (target_lat, target_lon)
                ).m

                if distance_m < 1000:
                    G.add_edge(
                        sensor_id, 
                        sensor_id_2, 
                        distance=distance_m, 
                        dir=direction
                    )

In [20]:
manual_edges = [(11,23), (9,22), (9,23), (1,12), (2,14), (3,14),(4,14), (5,15), (8,20), (6,19), (6,16), (8,19), (7,20), (7,19), (6,5), (17,14), (15,14), (16,14)]

In [21]:
import geopy.distance

node_positions = nx.get_node_attributes(G, "pos")

for source, target in manual_edges:
    lon1, lat1 = node_positions[source]
    lon2, lat2 = node_positions[target]

    distance_m = geopy.distance.geodesic((lat1, lon1), (lat2, lon2)).m
    G.add_edge(source, target, distance=distance_m)

In [22]:
sensor_ids = list(G.nodes())
node_map = {sid: idx for idx, sid in enumerate(sensor_ids)}
num_nodes = len(sensor_ids)

sample_df = list(traffic_dic[sensor_ids[0]].values())[0]

exclude_cols = ['latitude', 'longitude', 'timestamp', 'time', 'miles', "avtime", "hour"]
base_features = [col for col in sample_df.columns if col not in exclude_cols]

time_features = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos']
all_features = base_features + time_features

num_timestamps = len(sample_df)
num_features = len(all_features)

x_tensor = torch.zeros((num_nodes, num_features, num_timestamps), dtype=torch.float)

for sensor_id in sensor_ids:
    node_idx = node_map[sensor_id]
    traffic_df = list(traffic_dic[sensor_id].values())[0].copy()
    
    traffic_df['timestamp'] = pd.to_datetime(traffic_df['timestamp'])
    
    traffic_df['hour_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['hour_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['day_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df['day_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df["count_imputed"] = traffic_df["count_imputed"].apply(lambda x: 0 if x == False else 1)
    traffic_features_matrix = traffic_df[all_features].values.T
    x_tensor[node_idx, :, :] = torch.tensor(traffic_features_matrix, dtype=torch.float)

edge_list = []
edge_attr_list = []
for u, v, data in G.edges(data=True):
    edge_list.append([node_map[u], node_map[v]])
    edge_attr_list.append([data['distance']])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)

pyg_dataset = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr)

In [23]:
import plotly.graph_objects as go

num_nodes = len(node_map)
coords_traffic = np.zeros((num_nodes, 2))

for sensor_id, node_idx in node_map.items():
    lon, lat = G.nodes[sensor_id]['pos']
    coords_traffic[node_idx, 0] = lon
    coords_traffic[node_idx, 1] = lat

edge_t_to_t = pyg_dataset.edge_index


def visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t, mapbox_style="carto-positron"):
    """
    Visualizes the traffic sensor spatial graph directly on a map using Plotly Scattermapbox.
    """
    fig = go.Figure()

    # --- DRAW TRAFFIC-TO-TRAFFIC EDGES ---
    t_edge_lon, t_edge_lat = [], []
    t_start_nodes = edge_t_to_t[0].numpy()
    t_end_nodes = edge_t_to_t[1].numpy()
    
    for src, dst in zip(t_start_nodes, t_end_nodes):
        # Plotly draws continuous paths; adding None breaks the line between distinct edges
        t_edge_lon.extend([coords_traffic[src, 0], coords_traffic[dst, 0], None])
        t_edge_lat.extend([coords_traffic[src, 1], coords_traffic[dst, 1], None])
        
    fig.add_trace(go.Scattermapbox(
        lon=t_edge_lon, lat=t_edge_lat,
        mode='lines',
        line=dict(width=1.5, color='rgba(50, 150, 250, 0.6)'),
        name='Traffic-to-Traffic Edges',
        hoverinfo='none'
    ))

    fig.add_trace(go.Scattermapbox(
        lon=coords_traffic[:, 0], lat=coords_traffic[:, 1],
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.85),
        name='Traffic Sensor Nodes',
        text=[f"Node Index: {i}<br>Sensor ID: {sensor_ids[i]}" for i in range(len(coords_traffic))],
        hoverinfo='text'
    ))

    center_lat = np.mean(coords_traffic[:, 1])
    center_lon = np.mean(coords_traffic[:, 0])

    fig.update_layout(
        title=dict(text='Spatio-Temporal Traffic Graph Topology', font=dict(size=18)),
        autosize=True,
        hovermode='closest',
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.7)"),
        mapbox=dict(
            style=mapbox_style,
            bearing=0,
            center=dict(lat=center_lat, lon=center_lon),
            pitch=0,
            zoom=12
        ),
        width=1100,
        height=750,
        margin=dict(r=0, t=40, l=0, b=0)
    )
    
    fig.show()

visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t)

C:\Users\lucch\AppData\Local\Temp\ipykernel_1188\3245259071.py:30: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
C:\Users\lucch\AppData\Local\Temp\ipykernel_1188\3245259071.py:38: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Dropout, Layer, MultiHeadAttention, 
    Flatten, Reshape, Concatenate, Conv2D
)

class GraphConvLayer(Layer):
    """
    Custom Spatial Graph Convolution Layer processing 4D Tensors.
    Input shape:  (Batch, Timesteps, Nodes, Features)
    Output shape: (Batch, Timesteps, Nodes, Units)
    """
    def __init__(self, units, **kwargs):
        super(GraphConvLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer="glorot_uniform",
            trainable=True,
            name="gcn_weight"
        )
        super(GraphConvLayer, self).build(input_shape)

    def call(self, inputs, adj):
        transformed = tf.matmul(inputs, self.w)
        out = tf.einsum('ij,btjc->btic', adj, transformed)
        return tf.nn.relu(out)


# --- CONFIGURATION HYPERPARAMETERS ---
forecast_steps = 12
num_nodes = 24
traffic_features = 10
weather_features = 6
timesteps = 48

# ==========================================
# BRANCH 1: Spatio-Temporal Graph Network (STGCN)
# ==========================================
traffic_input = Input(shape=(timesteps, num_nodes, traffic_features), name="traffic_spatial_input")
adj_input = Input(shape=(num_nodes, num_nodes), name="adjacency_matrix", batch_size=1) 

# Temporal Gated Convolution Step 1
t_conv1 = Conv2D(filters=64, kernel_size=(3, 1), padding='same', activation='relu')(traffic_input)
t_conv1 = Dropout(0.2)(t_conv1)

# Spatial Graph Convolution Step 
s_gcn = GraphConvLayer(units=64)(t_conv1, adj_input[0])
s_gcn = Dropout(0.2)(s_gcn)

# Temporal Gated Convolution Step 2
t_conv2 = Conv2D(filters=32, kernel_size=(3, 1), padding='same', activation='relu')(s_gcn)
t_conv2 = Dropout(0.2)(t_conv2)

# Flatten Spatial-Temporal characteristics into a 1D embedding vector
flat_stgcn = Flatten()(t_conv2)


# ==========================================
# BRANCH 2: Sequential Base LSTM Base
# ==========================================
weather_input = Input(shape=(timesteps, weather_features), name="weather_temporal_input")

lstm_1 = LSTM(256, return_sequences=True)(weather_input)
drop_1 = Dropout(0.2)(lstm_1)

lstm_2 = LSTM(128, return_sequences=True)(drop_1) 
drop_2 = Dropout(0.2)(lstm_2)

lstm_3 = LSTM(64, return_sequences=True)(drop_2) 
drop_3 = Dropout(0.2)(lstm_3)

lstm_4 = LSTM(32, return_sequences=True)(drop_3) 
drop_4 = Dropout(0.2)(lstm_4)

# Attention filtering over LSTM outputs
mha = MultiHeadAttention(num_heads=4, key_dim=32)
attention_out = mha(drop_4, drop_4, drop_4)
flat_lstm = Flatten()(attention_out)


# ==========================================
# FUSION & REGRESSION HEAD
# ==========================================
merged_features = Concatenate()([flat_stgcn, flat_lstm])

dense_1 = Dense(400, activation="relu")(merged_features)
drop_5 = Dropout(0.2)(dense_1)

dense_2 = Dense(200, activation="relu")(drop_5)
drop_6 = Dropout(0.2)(dense_2)

dense_3 = Dense(50, activation="relu")(drop_6)
outputs = Dense(forecast_steps, name="forecast_prediction")(dense_3)


# Define and Compile Network Multi-Input Map
model = Model(inputs=[traffic_input, adj_input, weather_input], outputs=outputs)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

model.summary()

In [ ]:
import numpy as np

# 1. Transform PyG Edge Index into a Normalized Spatial Matrix (A)
nodes_count = pyg_dataset.x.shape[0]
raw_matrix = np.zeros((nodes_count, nodes_count), dtype=np.float32)
edges = pyg_dataset.edge_index.numpy()
raw_matrix[edges[0], edges[1]] = 1.0

# Symmetric normalization transformation: D^-0.5 * (A + I) * D^-0.5
raw_matrix += np.eye(nodes_count, dtype=np.float32)
degrees = np.sum(raw_matrix, axis=1)
deg_inv_sqrt = np.power(degrees, -0.5, where=degrees > 0)
deg_inv_sqrt[degrees == 0] = 0.0
D_inv = np.diag(deg_inv_sqrt)
normalized_adj_matrix = D_inv @ raw_matrix @ D_inv

# 2. Reshape and tile inputs to fit target multi-inputs
# Expected traffic structure for fit: (Samples, Timesteps, Nodes, Features)
# Expected adjacency input structure: (Samples, Nodes, Nodes)
# Expected exogenous sequence structure: (Samples, Timesteps, Features)

# Dummy variables below demonstrate how arrays must align:
num_samples = 1000

train_traffic_x = np.random.randn(num_samples, timesteps, num_nodes, traffic_features)
train_weather_x = np.random.randn(num_samples, timesteps, weather_features)
train_y = np.random.randn(num_samples, forecast_steps)

# Tile the static graph adjacency structure along the batch dimension
train_adj_x = np.repeat(np.expand_dims(normalized_adj_matrix, axis=0), num_samples, axis=0)

# Execute the multi-input tracking sequence
model.fit(
    x={
        "traffic_spatial_input": train_traffic_x,
        "adjacency_matrix": train_adj_x,
        "weather_temporal_input": train_weather_x
    },
    y=train_y,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)